# Crane PID AI — Data Exploration & Model Training

Notebook for exploring test_results.csv and training the PID predictor model.
Run cells in order.


In [ ]:
# Section 0 — Dataset picker
# Lists all available experiment datasets and sets DATA_PATH.
# Change the index or set DATA_PATH manually before running the rest of the notebook.
import glob

EXPERIMENTS_DIR = '../data/experiments'
datasets = sorted(glob.glob(f'{EXPERIMENTS_DIR}/*/model_data.csv'))

print("Available datasets:")
for i, path in enumerate(datasets):
    rows = sum(1 for _ in open(path)) - 1
    print(f"  [{i}] {path}  ({rows} rows)")

# Change index here to select a different dataset:
DATA_PATH = datasets[0]
print(f"\nUsing: {DATA_PATH}")


In [ ]:
# Section 1 — Load and explore data
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv(DATA_PATH)
print(f"Dataset: {DATA_PATH}")
print(f"Total rows: {len(df)}")
print(f"Status counts:\n{df['status'].value_counts()}\n")
print(df.describe())


In [ ]:
# Section 2 — Score and metric distributions
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
df['score'].hist(ax=axes[0], bins=50, color='teal', edgecolor='black')
axes[0].set_title('Score distribution (lower = better)')

df['t_settle'].hist(ax=axes[1], bins=50, color='steelblue', edgecolor='black')
axes[1].set_title('Settle time [s]')

df['overshoot_deg'].hist(ax=axes[2], bins=50, color='coral', edgecolor='black')
axes[2].set_title('Overshoot [°]')

plt.tight_layout()
plt.show()


In [ ]:
# Section 3 — Heatmap: Kp vs Kd for a specific scenario
top = df[df['status'] == 'ok'].nsmallest(50, 'score')
print(top[['L', 'm', 'Kp', 'Ki', 'Kd', 't_settle', 'overshoot_deg', 'score']].head(10))

pivot = df[(df['L'] == 10) & (df['m'] == 50)].pivot_table(
    values='score', index='Kd', columns='Kp', aggfunc='min')

if not pivot.empty:
    plt.figure(figsize=(10, 6))
    sns.heatmap(pivot, cmap='RdYlGn_r', annot=True, fmt='.2f', linewidths=0.5)
    plt.title('Score: Kp vs Kd (L=10m, m=50kg) — green=better')
    plt.show()
else:
    print('No data for L=10, m=50 — adjust filters')


In [ ]:
# Section 4 — Train the model
import sys
sys.path.insert(0, '../ai-service')
from model import PIDPredictor

p = PIDPredictor()
stats = p.train(DATA_PATH)

print(f"\nTraining complete")
print(f"  Dataset:          {DATA_PATH}")
print(f"  Rows used:        {stats['n_total']}")
print(f"  Trained at:       {stats['trained_at']}")
print(f"  Avg R²:           {stats.get('avg_r2', '—')}")
print(f"  Confidence:       {stats.get('confidence_label', '—')}")
dr = stats.get('data_range', {})
if dr:
    print(f"  Data range:       L=[{dr.get('L_min')}, {dr.get('L_max')}]  "
          f"m=[{dr.get('m_min')}, {dr.get('m_max')}]  "
          f"wind≤{dr.get('wind_max')}")
print()
for param, m in stats['metrics'].items():
    stars = '★' * int(m['r2'] * 5)
    print(f"  {param}: R²={m['r2']:.3f} {stars}  MAE={m['mae']:.4f}  "
          f"(train={m['n_train']}, test={m['n_test']})")


In [ ]:
# Section 5 — Validate predictions on sample scenarios
cases = [
    {'L': 5,  'm': 30,  'wind_speed': 5,  'wind_dir_deg': 45},
    {'L': 10, 'm': 50,  'wind_speed': 8,  'wind_dir_deg': 135},
    {'L': 15, 'm': 100, 'wind_speed': 12, 'wind_dir_deg': 270},
    {'L': 20, 'm': 200, 'wind_speed': 15, 'wind_dir_deg': 0},
]

header = f"{'L':>4} {'m':>5} {'Wind':>5} | {'Kp':>7} {'Ki':>7} {'Kd':>7} | {'Conf':>5} {'Label':<12} {'InRange'} {'Fallback'} {'Model'}"
print(header)
print('-' * len(header))
for c in cases:
    r = p.predict(**c)
    print(f"{c['L']:>4} {c['m']:>5} {c['wind_speed']:>5} | "
          f"{r['Kp']:>7.2f} {r['Ki']:>7.3f} {r['Kd']:>7.2f} | "
          f"{r['confidence']:>5.3f} {r.get('confidence_label','—'):<12} "
          f"{'yes' if r.get('in_training_range') else 'no ':>7}  "
          f"{'yes' if r.get('fallback') else 'no ':>8}  "
          f"{r['model']}")


In [ ]:
# Section 7 — Data generation (optional)
# Runs generate_optimal_pid.py --quick to regenerate the selected dataset.
# This overwrites DATA_PATH with fresh data, then you can re-run cells 1–6.
import subprocess

print(f"Regenerating: {DATA_PATH}")
result = subprocess.run(
    ['python', '../ai-service/generate_optimal_pid.py', '--quick', '--out', DATA_PATH],
    capture_output=True, text=True
)
print(result.stdout[-3000:])
if result.returncode != 0:
    print("STDERR:", result.stderr[-1000:])


In [ ]:
# Section 8 — Train model on new data and save it to model_dataset_auto directory
!pwd
!python ../ai-service/trainer.py ../data/experiments/model_dataset_auto/model_data.csv
!mv model.joblib ../data/experiments/model_dataset_auto/model.joblib

In [ ]:
# Section 8 — AI service status check
# Checks if the FastAPI service on port 8000 is running and trained.
import requests

try:
    r = requests.post('http://localhost:8000/train', json={
        "csv_path":   "../data/experiments/model_dataset_auto/model_data.csv",
        "output_dir": "../data/experiments/model_dataset_auto",
        "model_id":   "model_dataset_auto"
    }, timeout=60)
    #print(r.json())
    s = r.json()
    print(f"Trained:          {s.get('trained')}")
    print(f"Avg R²:           {s.get('avg_r2', s.get('stats', {}).get('avg_r2', '—'))}")
    print(f"Confidence:       {s.get('confidence_label', s.get('stats', {}).get('confidence_label', '—'))}")
    tr = s.get('training_range', {})
    if tr:
        print(f"Training range:   L=[{tr.get('L_min')}, {tr.get('L_max')}]  "
              f"m=[{tr.get('m_min')}, {tr.get('m_max')}]  "
              f"wind≤{tr.get('wind_max')}")
    r2d = s.get('r2_detail', {})
    if r2d:
        print(f"R² detail:        Kp={r2d.get('Kp','—')}  Ki={r2d.get('Ki','—')}  Kd={r2d.get('Kd','—')}")
except requests.exceptions.ConnectionError:
    print("AI service offline (start with: cd ai-service && python main.py)")
except Exception as e:
    print(f"Error: {e}")

In [ ]:
# Section 6 — Feature importances
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, target in zip(axes, ['Kp', 'Ki', 'Kd']):
    imp = p.models[target]['model'].feature_importances_
    colors = ['steelblue'] * len(PIDPredictor.FEATURE_COLS)
    ax.barh(PIDPredictor.FEATURE_COLS, imp, color=colors, edgecolor='black')
    ax.set_title(f'Feature importance — {target}')
    ax.set_xlabel('Importance')
plt.tight_layout()
plt.show()
